In [22]:
import os
import json
import requests
import pandas as pd

In [23]:
request_url = 'https://pokeapi.co/api/v2/generation/1'
response = requests.get(request_url)
data = response.json()
with open ('../../data/Pokemon Data/generation_1.json', 'w') as file: 
    json.dump(data,file)

gen_1_poke_df = pd.json_normalize(data, record_path = 'pokemon_species')
display(gen_1_poke_df)

,name,url
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/
1,charmander,https://pokeapi.co/api/v2/pokemon-species/4/
2,squirtle,https://pokeapi.co/api/v2/pokemon-species/7/
3,caterpie,https://pokeapi.co/api/v2/pokemon-species/10/
4,weedle,https://pokeapi.co/api/v2/pokemon-species/13/
...,...,...
146,dragonair,https://pokeapi.co/api/v2/pokemon-species/148/
147,dragonite,https://pokeapi.co/api/v2/pokemon-species/149/
148,mew,https://pokeapi.co/api/v2/pokemon-species/151/
149,ivysaur,https://pokeapi.co/api/v2/pokemon-species/2/


In [24]:
def extract_pokemon_id(url): 
    id = url.rstrip('/').split('/')[-1]
    return int(id)

In [25]:
gen_1_poke_df['pokemon_id'] = gen_1_poke_df['url'].apply(extract_pokemon_id)
display(gen_1_poke_df)
gen_1_poke_df = gen_1_poke_df.sort_values('pokemon_id').reset_index(drop = True)


,name,url,pokemon_id
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/,1
1,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,4
2,squirtle,https://pokeapi.co/api/v2/pokemon-species/7/,7
3,caterpie,https://pokeapi.co/api/v2/pokemon-species/10/,10
4,weedle,https://pokeapi.co/api/v2/pokemon-species/13/,13
...,...,...,...
146,dragonair,https://pokeapi.co/api/v2/pokemon-species/148/,148
147,dragonite,https://pokeapi.co/api/v2/pokemon-species/149/,149
148,mew,https://pokeapi.co/api/v2/pokemon-species/151/,151
149,ivysaur,https://pokeapi.co/api/v2/pokemon-species/2/,2


In [35]:
# THIS FUNCTION IS VERY INEFFICIENT - THIS VERSION IS FOR TESTING ONLY
def get_pokemon_details(pokemon_id): 
    request_url = f'https://pokeapi.co/api/v2/pokemon/{pokemon_id}'
    response = requests.get(request_url)
    data = response.json()
    if response.status_code != 200:
        print(f"Error: Received status code {response.status_code}")
        print(f"Response content: {response.text}")
    else:
        with open(f'../../data/Pokemon Data/generation_1_pokemon_details/{pokemon_id}_species_details', 'w') as file:
            json.dump(data, file)
            species_details_df = pd.json_normalize(data)
            species_details_df.drop(columns = [col for col in species_details_df.columns if "sprites" in col], inplace = True)
    return species_details_df

test_df = get_pokemon_details(1)
print(test_df.columns.tolist())
display(test_df['stats'][0])

['abilities', 'base_experience', 'forms', 'game_indices', 'height', 'held_items', 'id', 'is_default', 'location_area_encounters', 'moves', 'name', 'order', 'past_abilities', 'past_types', 'stats', 'types', 'weight', 'cries.latest', 'cries.legacy', 'species.name', 'species.url']


[{'base_stat': 45,
  'effort': 0,
  'stat': {'name': 'hp', 'url': 'https://pokeapi.co/api/v2/stat/1/'}},
 {'base_stat': 49,
  'effort': 0,
  'stat': {'name': 'attack', 'url': 'https://pokeapi.co/api/v2/stat/2/'}},
 {'base_stat': 49,
  'effort': 0,
  'stat': {'name': 'defense', 'url': 'https://pokeapi.co/api/v2/stat/3/'}},
 {'base_stat': 65,
  'effort': 1,
  'stat': {'name': 'special-attack',
   'url': 'https://pokeapi.co/api/v2/stat/4/'}},
 {'base_stat': 65,
  'effort': 0,
  'stat': {'name': 'special-defense',
   'url': 'https://pokeapi.co/api/v2/stat/5/'}},
 {'base_stat': 45,
  'effort': 0,
  'stat': {'name': 'speed', 'url': 'https://pokeapi.co/api/v2/stat/6/'}}]

def get_pokemon_species_type(pokemon_id): 
    request_url = f'https://pokeapi.co/api/v2/type/{pokemon_id}'
    response = requests.get(request_url)
    data = response.json()
    if response.status_code != 200:
        print(f"Error: Received status code {response.status_code}")
        print(f"Response content: {response.text}")
    else:
        with open(f'../data/../generation_1_pokemon_type_details/{pokemon_id}_type_details', 'w') as file:
            json.dump(data, file)
            species_type_df = pd.json_normalize(data)
    return species_type_df

get_pokemon_species_type(1)

In [ ]:
list_of_species_details_df = gen_1_poke_df['pokemon_id'].apply(get_pokemon_species_details)
species_details_df = pd.concat(list_of_species_details_df.to_list(), ignore_index=True)

list_of_species_type_df = gen_1_poke_df['pokemon_id'].apply(get_pokemon_species_type)
species_type_df = pd.concat(list_of_species_type_df.to_list(), ignore_index=True)

In [ ]:
species_details_df['habitat.name'].value_counts()